
# Notebook para generar insumos del Capítulo 4

Esta notebook está pensada para **extraer, resumir y exportar** los datos que necesitás para el **Capítulo 4: Ensayos y Resultados** de la memoria.

Genera automáticamente insumos para:

- **4.1 Configuración experimental**
- **4.2 Resultados de los modelos predictivos**
- **4.3 Impacto en la optimización de portafolios**
- **4.4 Discusión de resultados**

## Qué produce

Dentro de `outputs/chapter4_inputs/` guarda:

- tablas `.csv` listas para usar o adaptar en LaTeX,
- figuras `.png` para el capítulo,
- comparaciones entre escenarios clásico, MLP y LSTM.

## Cómo usarla

1. Colocá esta notebook dentro del repo **Markowitz** o en una subcarpeta del repo.
2. Corré las celdas de arriba hacia abajo.
3. Si la detección automática de la raíz del repo falla, completá `ROOT_HINT` con la ruta local correspondiente.


In [ ]:

from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 140)
pd.set_option('display.float_format', lambda x: f"{x:,.6f}")


## 1. Parámetros de ejecución

In [ ]:

# Si la detección automática falla, completá esta ruta con la raíz del repo Markowitz.
ROOT_HINT = None

# Carpeta donde se guardarán tablas y figuras generadas por esta notebook.
EXPORT_SUBDIR = 'outputs/chapter4_inputs'

# Ticker y modelo para el gráfico de retornos reales vs predichos de la sección 4.2.
TICKER_PLOT = 'AAPL'
MODEL_TO_PLOT = 'lstm'   # opciones: 'mlp' o 'lstm'

# Si querés incluir también el escenario de volatilidad en comparaciones adicionales.
INCLUDE_AI_VOLATILITY = False


## 2. Detección de la raíz del repositorio

In [ ]:
def find_repo_root(root_hint=None):
    def is_repo(p: Path) -> bool:
        return (p / 'Data' / 'returns.csv').exists() and (p / 'outputs').exists()

    if root_hint is not None:
        p = Path(root_hint).expanduser().resolve()
        if is_repo(p):
            return p

    start = Path.cwd().resolve()
    candidates = [start, *start.parents]

    for p in candidates:
        if is_repo(p):
            return p
        if is_repo(p / 'Markowitz'):
            return p / 'Markowitz'

        # Busca también uno o dos niveles hacia abajo por si la notebook quedó fuera del repo.
        for child in list(p.glob('*')) + list(p.glob('*/*')):
            if child.is_dir() and is_repo(child):
                return child

    raise FileNotFoundError(
        'No se encontró la raíz del repo Markowitz. '
        'Definí ROOT_HINT con la ruta correspondiente.'
    )

ROOT = find_repo_root(ROOT_HINT)
EXPORT_DIR = ROOT / EXPORT_SUBDIR
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

print('ROOT =', ROOT)
print('EXPORT_DIR =', EXPORT_DIR)


## 3. Carga de datos base y outputs del proyecto

In [ ]:

returns = pd.read_csv(ROOT / 'Data' / 'returns.csv')
returns['Date'] = pd.to_datetime(returns['Date'])
assets = [c for c in returns.columns if c != 'Date']

metadata = pd.read_csv(ROOT / 'Data' / 'assets_metadata.csv')

mlp_latest = json.loads((ROOT / 'outputs' / 'mlp_lab' / 'latest_run.json').read_text())
lstm_latest = json.loads((ROOT / 'outputs' / 'lstm_lab' / 'latest_run.json').read_text())

mlp_run_dir = ROOT / 'outputs' / 'mlp_lab' / 'runs' / mlp_latest['run_id']
lstm_run_dir = ROOT / 'outputs' / 'lstm_lab' / 'runs' / lstm_latest['run_id']

mlp_config = json.loads((mlp_run_dir / 'config.json').read_text())
mlp_summary = json.loads((mlp_run_dir / 'summary.json').read_text())

lstm_config = json.loads((lstm_run_dir / 'config.json').read_text())
lstm_summary = json.loads((lstm_run_dir / 'summary.json').read_text())

print('Run MLP  :', mlp_latest['run_id'])
print('Run LSTM :', lstm_latest['run_id'])
print('Observaciones totales:', len(returns))
print('Activos:', len(assets))


## 4. Helpers

In [ ]:

def get_split_counts(run_dir, ticker='AAPL'):
    file_path = run_dir / 'per_ticker' / f'{ticker}_predictions.csv'
    if not file_path.exists():
        tickers = sorted([p.name.replace('_predictions.csv', '') for p in (run_dir / 'per_ticker').glob('*_predictions.csv')])
        ticker = tickers[0]
        file_path = run_dir / 'per_ticker' / f'{ticker}_predictions.csv'

    df = pd.read_csv(file_path)
    counts = df['split'].value_counts().to_dict()
    return {
        'train': int(counts.get('train', 0)),
        'val': int(counts.get('val', 0)),
        'test': int(counts.get('test', 0)),
    }


def compute_prediction_metrics(run_dir, model_name):
    rows = []
    for file_path in sorted((run_dir / 'per_ticker').glob('*_predictions.csv')):
        ticker = file_path.name.replace('_predictions.csv', '')
        df = pd.read_csv(file_path)
        test = df[df['split'] == 'test'].copy()
        if test.empty:
            continue

        y_true = test['y_true'].to_numpy()
        y_pred = test['y_pred'].to_numpy()

        rmse = float(np.sqrt(np.mean((y_true - y_pred) ** 2)))
        mae = float(np.mean(np.abs(y_true - y_pred)))
        dir_acc = float(np.mean(np.sign(y_true) == np.sign(y_pred)))
        corr = float(pd.Series(y_true).corr(pd.Series(y_pred))) if len(test) > 1 else np.nan

        rows.append({
            'modelo': model_name,
            'ticker': ticker,
            'n_test': len(test),
            'rmse': rmse,
            'mae': mae,
            'dir_acc': dir_acc,
            'corr': corr,
        })
    return pd.DataFrame(rows)


def load_metric_table(path, label):
    df = pd.read_csv(path)
    df.insert(0, 'escenario', label)
    return df


def load_weights_table(path, label, metadata_df):
    df = pd.read_csv(path)
    df.insert(0, 'escenario', label)
    df = df.merge(metadata_df, on='ticker', how='left')
    return df


def concentration_stats(weights_df):
    w = weights_df['weight'].to_numpy()
    return {
        'hhi': float(np.sum(w ** 2)),
        'n_activos_con_peso': int(np.sum(w > 1e-8)),
        'n_activos_peso_mayor_1pct': int(np.sum(w >= 0.01)),
        'peso_maximo': float(np.max(w)),
    }


# 4.1 Configuración experimental

In [ ]:

dataset_summary = pd.DataFrame([{
    'fecha_inicio': returns['Date'].min().date().isoformat(),
    'fecha_fin': returns['Date'].max().date().isoformat(),
    'n_observaciones': int(len(returns)),
    'n_activos': int(len(assets)),
    'n_bloques': int(metadata['block'].nunique()),
}])

split_df = pd.DataFrame([
    {'modelo': 'MLP', **get_split_counts(mlp_run_dir, TICKER_PLOT)},
    {'modelo': 'LSTM', **get_split_counts(lstm_run_dir, TICKER_PLOT)},
])

config_summary = pd.DataFrame([
    {
        'modelo': 'MLP',
        'start_date': mlp_config.get('start_date'),
        'forecast_horizon': mlp_config.get('forecast_horizon'),
        'target_aggregation': mlp_config.get('target_aggregation'),
        'lags': mlp_config.get('lags'),
        'seq_len': np.nan,
        'hidden_layers': str(mlp_config.get('hidden_layers')),
        'hidden_size': np.nan,
        'epochs_o_max_iter': mlp_config.get('max_iter'),
    },
    {
        'modelo': 'LSTM',
        'start_date': lstm_config.get('start_date'),
        'forecast_horizon': lstm_config.get('forecast_horizon'),
        'target_aggregation': lstm_config.get('target_aggregation'),
        'lags': np.nan,
        'seq_len': lstm_config.get('seq_len'),
        'hidden_layers': np.nan,
        'hidden_size': lstm_config.get('hidden_size'),
        'epochs_o_max_iter': lstm_config.get('epochs'),
    },
])

dataset_summary.to_csv(EXPORT_DIR / 'tabla_4_1_dataset.csv', index=False)
metadata.sort_values(['block', 'asset_class', 'ticker']).to_csv(EXPORT_DIR / 'tabla_4_1_activos.csv', index=False)
split_df.to_csv(EXPORT_DIR / 'tabla_4_1_splits.csv', index=False)
config_summary.to_csv(EXPORT_DIR / 'tabla_4_1_config_modelos.csv', index=False)

dataset_summary


In [ ]:
split_df

In [ ]:
metadata.sort_values(['block', 'asset_class', 'ticker']).head(15)


### Archivos exportados para 4.1

- `tabla_4_1_dataset.csv`
- `tabla_4_1_activos.csv`
- `tabla_4_1_splits.csv`
- `tabla_4_1_config_modelos.csv`


# 4.2 Resultados de los modelos predictivos

In [ ]:

mlp_metrics = compute_prediction_metrics(mlp_run_dir, 'MLP')
lstm_metrics = compute_prediction_metrics(lstm_run_dir, 'LSTM')
all_metrics = pd.concat([mlp_metrics, lstm_metrics], ignore_index=True)
summary_metrics = all_metrics.groupby('modelo')[['rmse', 'mae', 'dir_acc', 'corr']].mean().reset_index()

all_metrics.to_csv(EXPORT_DIR / 'tabla_4_2_metricas_por_activo.csv', index=False)
summary_metrics.to_csv(EXPORT_DIR / 'tabla_4_2_metricas_resumen.csv', index=False)

summary_metrics


In [ ]:

# Top 10 activos por directional accuracy para cada modelo
all_metrics.sort_values(['modelo', 'dir_acc'], ascending=[True, False]).groupby('modelo').head(10)


In [ ]:

# Gráfico de retornos reales vs predichos para el ticker/modelo elegido
run_dir = lstm_run_dir if MODEL_TO_PLOT.lower() == 'lstm' else mlp_run_dir
plot_file = run_dir / 'per_ticker' / f'{TICKER_PLOT}_predictions.csv'
plot_df = pd.read_csv(plot_file)
plot_df['date'] = pd.to_datetime(plot_df['date'])
plot_df = plot_df[plot_df['split'] == 'test'].copy()

plt.figure(figsize=(12, 5))
plt.plot(plot_df['date'], plot_df['y_true'], label='Real')
plt.plot(plot_df['date'], plot_df['y_pred'], label='Predicho')
plt.title(f'Retornos reales vs predichos - {TICKER_PLOT} ({MODEL_TO_PLOT.upper()})')
plt.xlabel('Fecha')
plt.ylabel('Retorno log diario')
plt.legend()
plt.tight_layout()
plt.savefig(EXPORT_DIR / f'figura_4_2_reales_vs_predichos_{TICKER_PLOT}_{MODEL_TO_PLOT}.png', dpi=200)
plt.show()



### Archivos exportados para 4.2

- `tabla_4_2_metricas_por_activo.csv`
- `tabla_4_2_metricas_resumen.csv`
- `figura_4_2_reales_vs_predichos_<ticker>_<modelo>.png`


# 4.3 Impacto en la optimización de portafolios

In [ ]:

frontier_map = {
    'Clásico': ROOT / 'outputs' / 'frontier_classic.csv',
    'IA retornos (MLP)': ROOT / 'outputs' / 'frontier_ai_returns.csv',
    'IA retornos (LSTM)': ROOT / 'outputs' / 'frontier_ai_returns_lstm.csv',
}

if INCLUDE_AI_VOLATILITY:
    frontier_map['IA volatilidad'] = ROOT / 'outputs' / 'frontier_ai_volatility.csv'

plt.figure(figsize=(8, 6))
for label, path in frontier_map.items():
    df = pd.read_csv(path)
    plt.plot(df['vol_annual'], df['return_annual'], label=label)

plt.xlabel('Volatilidad anual')
plt.ylabel('Retorno anual esperado')
plt.title('Comparación de fronteras eficientes')
plt.legend()
plt.tight_layout()
plt.savefig(EXPORT_DIR / 'figura_4_3_fronteras_comparadas.png', dpi=200)
plt.show()


In [ ]:

metrics_map = {
    'Clásico': ROOT / 'outputs' / 'metrics' / 'metrics_classic.csv',
    'IA retornos (MLP)': ROOT / 'outputs' / 'metrics' / 'metrics_ai_returns.csv',
    'IA retornos (LSTM)': ROOT / 'outputs' / 'metrics' / 'metrics_ai_returns_lstm.csv',
}

portfolio_metrics = pd.concat(
    [load_metric_table(path, label) for label, path in metrics_map.items()],
    ignore_index=True,
)

portfolio_metrics.to_csv(EXPORT_DIR / 'tabla_4_3_metricas_portafolio.csv', index=False)
portfolio_metrics


In [ ]:

weights_map = {
    'Clásico': ROOT / 'outputs' / 'weights' / 'weights_maxsharpe_classic.csv',
    'IA retornos (MLP)': ROOT / 'outputs' / 'weights' / 'weights_maxsharpe_ai_returns.csv',
    'IA retornos (LSTM)': ROOT / 'outputs' / 'weights' / 'weights_maxsharpe_ai_returns_lstm.csv',
}

weights_long = pd.concat(
    [load_weights_table(path, label, metadata) for label, path in weights_map.items()],
    ignore_index=True,
)
weights_long.to_csv(EXPORT_DIR / 'tabla_4_3_pesos_largos.csv', index=False)

top_weights = weights_long.sort_values(['escenario', 'weight'], ascending=[True, False]).groupby('escenario').head(10)
top_weights.to_csv(EXPORT_DIR / 'tabla_4_3_top10_pesos.csv', index=False)

top_weights


In [ ]:

classic = pd.read_csv(weights_map['Clásico']).rename(columns={'weight': 'weight_classic'})
mlp = pd.read_csv(weights_map['IA retornos (MLP)']).rename(columns={'weight': 'weight_mlp'})
lstm = pd.read_csv(weights_map['IA retornos (LSTM)']).rename(columns={'weight': 'weight_lstm'})

weights_compare = (
    classic
    .merge(mlp, on='ticker')
    .merge(lstm, on='ticker')
    .merge(metadata, on='ticker', how='left')
)
weights_compare['delta_mlp_vs_classic'] = weights_compare['weight_mlp'] - weights_compare['weight_classic']
weights_compare['delta_lstm_vs_classic'] = weights_compare['weight_lstm'] - weights_compare['weight_classic']
weights_compare.to_csv(EXPORT_DIR / 'tabla_4_3_pesos_comparados.csv', index=False)

weights_compare.sort_values('delta_lstm_vs_classic', ascending=False).head(15)



### Archivos exportados para 4.3

- `figura_4_3_fronteras_comparadas.png`
- `tabla_4_3_metricas_portafolio.csv`
- `tabla_4_3_pesos_largos.csv`
- `tabla_4_3_top10_pesos.csv`
- `tabla_4_3_pesos_comparados.csv`


# 4.4 Insumos para la discusión de resultados

In [ ]:

base = pd.read_csv(weights_map['Clásico']).rename(columns={'weight': 'weight_classic'})
helper_rows = []

for label, path in weights_map.items():
    df = pd.read_csv(path)
    stats = concentration_stats(df)
    merged = base.merge(df.rename(columns={'weight': 'weight_current'}), on='ticker', how='outer').fillna(0.0)
    turnover_vs_classic = 0.5 * np.abs(merged['weight_current'] - merged['weight_classic']).sum()
    helper_rows.append({
        'escenario': label,
        **stats,
        'turnover_vs_classic': float(turnover_vs_classic),
    })

helper_df = pd.DataFrame(helper_rows)
helper_df.to_csv(EXPORT_DIR / 'tabla_4_4_indicadores_discusion.csv', index=False)
helper_df



### Cómo leer esta tabla

- **HHI**: mayor valor implica más concentración.
- **n_activos_peso_mayor_1pct**: cuántos activos tienen peso relevante.
- **peso_maximo**: identifica si el portafolio quedó demasiado cargado en un solo activo.
- **turnover_vs_classic**: cuánto cambia la cartera respecto al escenario clásico.


# Resumen final de archivos generados

In [ ]:
sorted([p.name for p in EXPORT_DIR.glob('*')])


## Sugerencia de uso en tu memoria

### Para 4.1
Usá principalmente:
- `tabla_4_1_dataset.csv`
- `tabla_4_1_activos.csv`
- `tabla_4_1_splits.csv`

### Para 4.2
Usá principalmente:
- `tabla_4_2_metricas_resumen.csv`
- `tabla_4_2_metricas_por_activo.csv`
- `figura_4_2_reales_vs_predichos_<ticker>_<modelo>.png`

### Para 4.3
Usá principalmente:
- `figura_4_3_fronteras_comparadas.png`
- `tabla_4_3_metricas_portafolio.csv`
- `tabla_4_3_top10_pesos.csv`
- `tabla_4_3_pesos_comparados.csv`

### Para 4.4
Usá principalmente:
- `tabla_4_4_indicadores_discusion.csv`
